In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/USW00014711_merged.csv", header=None)
df.head(5)

In [ ]:
df.columns = df.iloc[0]

In [ ]:
df = df.iloc[1:]
df.head(5)

In [ ]:
# just curious about my birthday's conditions in PA
df.loc[df["date"] == "2005-03-17", ["date", "TAVG", "era5_precip_sum", "era5_potential_snow_hours"]]

In [ ]:
# little bit of fun, let's look at total precip for each year
import matplotlib.pyplot as plt

df_precip = df[["date", "era5_precip_sum"]]
df_precip["date"] = pd.to_datetime(df_precip["date"])
df_precip["era5_precip_sum"] = pd.to_numeric(df_precip["era5_precip_sum"], errors='coerce')
df_yearly = df_precip.set_index('date').resample('YE')['era5_precip_sum'].sum()

plt.figure(figsize=(10, 6))
plt.bar(df_yearly.index.year, df_yearly.values)
plt.title('Total Precip by Year')
plt.show()

In [ ]:
# Or, if we wanted monthly instead...
import calendar

df_monthly_totals = df_precip.set_index('date').resample('ME')['era5_precip_sum'].sum()
df_seasonal_avg = df_monthly_totals.groupby(df_monthly_totals.index.month).mean()

plt.figure(figsize=(10, 6))
month_names = [calendar.month_abbr[i] for i in df_seasonal_avg.index]

plt.bar(month_names, df_seasonal_avg.values, color='teal', alpha=0.7)
plt.title('Average Monthly Precipitation (Seasonality)', fontsize=16)
plt.ylabel('Average Total Precip (mm/inches)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.show()

We need to add each of our station's dataframes together.

There is some debate about whether this introduces noise or not, but I believe the benefit of 6-xing our data size will outweight the potential detriment of using stations that may be at most 50-100 miles apart. Pennsylvania doesn't have huge variation in its climate across the state, so this shouldn't be a problem anyway.

In [ ]:
from pathlib import Path
file_path = Path("../data/processed/")
df_all = pd.DataFrame()
files = list(file_path.iterdir())
for file in files:
    print(file)
    # gitkeep file obviously can't be included in the dataframe
    if (".gitkeep" in file.as_posix()):
        continue
    df_curr = pd.read_csv(file)
    df_all = pd.concat([df_all, df_curr])

df_all.head(5)

In [ ]:
# copied and pasted collect.py function to avoid complicated import statement
AREA = [41.50, -78.50, 40.0, -76.50]

GHCN_STATION_METADATA_URL = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt"
def get_ghcn_stations(area):
    """parses GHCN metadata to find stations within bounds"""
    df = pd.read_fwf(GHCN_STATION_METADATA_URL, header=None,
                     widths=[11, 9, 10, 7, 3, 31],
                     names=["id", "lat", "lon", "elev", "state", "name"])
    north, west, south, east = area
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    print(f"lat: {df['lat']}")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    print(f"lon: {df['lon']}")
    mask = (df["lat"] <= north) & (df["lat"] >= south) & (df["lon"] <= east) & (df["lon"] >= west)
    return df[mask]

In [ ]:
# now that we've concatenated the dataframes, we also need the elevations for each ghcn station to help the ML model differentiate them

ghcn_df = get_ghcn_stations(AREA)
ghcn_df.head(5)

In [ ]:
# want to merge the elevation data with our large dataframe
cols_to_keep = df_all.columns.tolist() + ["elev_y"]
df_with_elev = pd.merge(df_all, ghcn_df, on="id", how="inner")
print(ghcn_df.columns)
print(df_with_elev.columns)
# df_with_elev = df_with_elev[cols_to_keep]
df_with_elev